In [2]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import mean_squared_error
from skimage.metrics import structural_similarity as ssim
from libpysal.weights import KNN
from esda.moran import Moran
import numpy as np

def moran_score(pred, true):
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    # 避免除零：当 true == 0 时，如果 pred == 0 返回1，否则返回0
    # 但按照原公式，如果 true == 0 且 pred <= true (即 pred <= 0)，只有 pred=0 时成立，0/0 是 nan。
    # 我们可以计算两个分支，然后选择。
    with np.errstate(divide='ignore', invalid='ignore'):
        branch1 = pred / true
        branch2 = (1 - pred) / (1 - true)
    result = np.where(pred <= true, branch1, branch2)
    # 处理 true == 0 的情况
    if np.any(true == 0):
        # 对于 true == 0 且 pred == 0，结果应为 1
        # 对于 true == 0 且 pred > 0，原公式会走 branch2，得到 (1-pred)/(1-0) = 1-pred，这不太合理。
        # 但通常 true 不会为0，我们可以额外处理使结果在 [0,1] 且符合直觉。
        # 为了简单，我们仅处理 pred=0, true=0 -> 1，其他 true=0 -> 0
        zero_true_mask = (true == 0)
        result = np.where(zero_true_mask & (pred == 0), 1.0, result)
        result = np.where(zero_true_mask & (pred != 0), 0.0, result)
    return result

def evaluate_deconvolution(pred, true, coords=None, eps=1e-12,
                           ssim_grid_size=64, knn_k=6):
    """
    Evaluate deconvolution prediction against ground truth.

    Parameters:
        pred : np.ndarray, shape (n_spots, n_types)
        true : np.ndarray, shape (n_spots, n_types)
        coords : np.ndarray, shape (n_spots, 2), optional
            Spatial coordinates of spots. Required for Moran's I and SSIM.
        eps : float
        ssim_grid_size : int
            Grid size for rasterizing spatial maps (e.g., 64 -> 64x64).
        knn_k : int
            Number of neighbors for Moran's I spatial weight matrix.

    Returns:
        pd.DataFrame with all metrics.
    """
    pred = np.asarray(pred)
    true = np.asarray(true)
    assert pred.shape == true.shape, "Shape mismatch"
    n_spots, n_types = pred.shape

    # ---- 1. Per-spot PCC / Spearman ----
    spot_pcc, spot_spearman = [], []
    for i in range(n_spots):
        p, t = pred[i, :], true[i, :]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            spot_pcc.append(pcc)
            spot_spearman.append(spearman)
    mean_spot_pcc = np.nanmean(spot_pcc) if spot_pcc else np.nan
    mean_spot_spearman = np.nanmean(spot_spearman) if spot_spearman else np.nan

    # ---- 2. Per-cell-type PCC / Spearman ----
    type_pcc, type_spearman = [], []
    for j in range(n_types):
        p, t = pred[:, j], true[:, j]
        if np.std(p) > eps and np.std(t) > eps:
            pcc, _ = pearsonr(p, t)
            spearman, _ = spearmanr(p, t)
            type_pcc.append(pcc)
            type_spearman.append(spearman)
    mean_type_pcc = np.nanmean(type_pcc) if type_pcc else np.nan
    mean_type_spearman = np.nanmean(type_spearman) if type_spearman else np.nan

    # ---- 3. MSE / RMSE ----
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)

    # ---- 4. Jensen-Shannon Divergence (per spot) ----
    jsd_list = []
    for i in range(n_spots):
        p = np.maximum(pred[i, :], 0)
        t = np.maximum(true[i, :], 0)
        if np.sum(p) < eps or np.sum(t) < eps:
            continue
        p = p / (np.sum(p) + eps)
        t = t / (np.sum(t) + eps)
        jsd = jensenshannon(p, t, base=2) ** 2
        jsd_list.append(jsd)
    mean_jsd = np.mean(jsd_list) if jsd_list else np.nan

    # ---- 5. Spatial metrics (SSIM + Moran's I) ----
    mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff = np.nan, np.nan, np.nan, np.nan
    if coords is not None:
        coords = np.asarray(coords)

        # 5a. SSIM per cell type (rasterize spatial map to a 2D grid)
        ssim_list = []
        for j in range(n_types):
            pred_grid = _rasterize(coords, pred[:, j], grid_size=ssim_grid_size)
            true_grid = _rasterize(coords, true[:, j], grid_size=ssim_grid_size)
            # Skip if either grid is constant (SSIM undefined)
            if np.std(pred_grid) < eps or np.std(true_grid) < eps:
                continue
            # Use a fixed data range for both
            data_range = max(pred_grid.max(), true_grid.max()) - min(pred_grid.min(), true_grid.min())
            if data_range < eps:
                continue
            s = ssim(pred_grid, true_grid, data_range=data_range)
            ssim_list.append(s)
        mean_ssim = np.nanmean(ssim_list) if ssim_list else np.nan

        # 5b. Moran's I per cell type
        try:
            w = KNN.from_array(coords, k=knn_k)
            w.transform = 'r'  # row-standardized
            moran_pred_list, moran_true_list = [], []
            for j in range(n_types):
                p = pred[:, j]
                t = true[:, j]
                if np.std(p) > eps:
                    mi_p = Moran(p, w).I
                    moran_pred_list.append(mi_p)
                if np.std(t) > eps:
                    mi_t = Moran(t, w).I
                    moran_true_list.append(mi_t)
            mean_moran_pred = np.nanmean(moran_pred_list) if moran_pred_list else np.nan
            mean_moran_true = np.nanmean(moran_true_list) if moran_true_list else np.nan
            if not np.isnan(mean_moran_pred) and not np.isnan(mean_moran_true):
                mean_moran_diff = moran_score(mean_moran_pred , mean_moran_true)
        except Exception as e:
            print(f"[Warning] Moran's I failed: {e}")

    # ---- Assemble results ----
    results = {
        'metric': [
            'mean_spot_pcc', 'mean_spot_spearman',
            'mean_type_pcc', 'mean_type_spearman',
            'overall_mse', 'overall_rmse', 'mean_jsd',
            'mean_ssim', 'mean_moran_pred', 'mean_moran_true', 'mean_moran_diff'
        ],
        'value': [
            mean_spot_pcc, mean_spot_spearman,
            mean_type_pcc, mean_type_spearman,
            mse, rmse, mean_jsd,
            mean_ssim, mean_moran_pred, mean_moran_true, mean_moran_diff
        ]
    }
    df = pd.DataFrame(results).set_index('metric')
    return df


def _rasterize(coords, values, grid_size=64):
    """
    Rasterize scattered spatial values onto a 2D grid using nearest-neighbor assignment.
    Empty grid cells are filled with 0.
    """
    coords = np.asarray(coords)
    values = np.asarray(values)
    x, y = coords[:, 0], coords[:, 1]
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-12)
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-12)
    xi = np.clip((x_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)
    yi = np.clip((y_norm * (grid_size - 1)).astype(int), 0, grid_size - 1)

    grid = np.zeros((grid_size, grid_size), dtype=float)
    count = np.zeros((grid_size, grid_size), dtype=float)
    np.add.at(grid, (yi, xi), values)
    np.add.at(count, (yi, xi), 1)
    # Average per grid cell
    grid = np.divide(grid, count, out=np.zeros_like(grid), where=count > 0)
    return grid

In [48]:
#ours
import scanpy as sc
import pandas as pd
import numpy as np
import pandas as pd
celltype = pd.read_csv('/data/qyyuan/project/ST_CP/JE/simulation/MERFISH/merge_df_mouse1sample1_mouse1_slice50.csv',sep=',',header = 0,index_col=0)

data0 = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/ours/merfish_A_optimized_0425.csv',sep=',',header = 0,index_col=0)
A_gt_deconv=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Merfish/A_gt_simulationdata.csv',sep=',',header=0,index_col=0)
spot_clean=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Merfish/spots_info.csv',sep=',',header=0,index_col=0)




In [49]:
spot_clean=spot_clean[['center_x','center_y']]

In [30]:
#ours = pd.read_csv('/home/qyyuan/project/ST_CP/JE/method/predicted_A_matrix-0425-BestModelLM1L30.1_novo_0.0001.csv',sep=',',index_col=0)
data0[data0<0.045*27/23]=0
row_sums = data0.sum(axis=1)
A_pred = data0.div(row_sums,axis=0)
df_results_ours = evaluate_deconvolution(A_pred.values, A_gt_deconv.values,coords=spot_clean)

In [32]:
# novosparc
novo = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/novosparc_normalization.npy') 
subclass=celltype['subclass'].values



df_onehot = pd.get_dummies(subclass) 
class_names = df_onehot.columns.tolist()
X = df_onehot.values.T 
print(f"矩阵 X 的形状: {X.shape}") # 应输出 (23, 4198)
result = np.dot(X, novo)
result=result.T
row_sums = result.sum(axis=1)  # 每个 spot 的总和，形状 (424,)
spot_ct_proportions = result / row_sums[:, np.newaxis]  # 或者 result / row_sums.reshape(-1,1)
df_results_novosparc = evaluate_deconvolution(spot_ct_proportions, A_gt_deconv.values,coords=spot_clean)

矩阵 X 的形状: (23, 4198)


In [34]:
class_names==A_gt_deconv.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [35]:
# tangram
spatial_with_tangram=sc.read("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spatial_with_tangram-Simulation.h5ad")
tangram=spatial_with_tangram.obsm["tangram_ct_pred"]
tangram=tangram[class_names].values
tangram=tangram/tangram.sum(axis=1,keepdims=True)
df_results_tangram = evaluate_deconvolution(tangram, A_gt_deconv.values,coords=spot_clean)


In [36]:
#spotiphy
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/compare/spotiphy/Spotiphy_Result-SimulationData/proportion2-simulationdata.csv',header=0,index_col=0)
A_dev_pred=A_dev_pred.loc[A_gt_deconv.index]
A_dev_pred=A_dev_pred[A_gt_deconv.columns]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
df_results_spotiphy = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean)

In [40]:
A_dev_pred

,Astro,Endo,L2/3 IT,L4/5 IT,L5 ET,L5 IT,L5/6 NP,L6 CT,L6 IT,L6 IT Car3,...,OPC,Oligo,PVM,Peri,Pvalb,SMC,Sncg,Sst,VLMC,Vip
spot_id,,,,,,,,,,,,,,,,,,,,,
0,0.000068,0.003427,0.000143,0.000176,0.000087,0.000181,0.000109,0.000090,0.000122,0.431125,...,0.000077,0.000063,0.000285,0.215719,0.000088,0.342127,0.005150,0.000082,0.000385,0.000114
1,0.166615,0.638544,0.000095,0.000117,0.000086,0.000140,0.000083,0.000094,0.000118,0.049348,...,0.000076,0.000077,0.000192,0.000058,0.000160,0.142869,0.000096,0.000778,0.000096,0.000082
2,0.534989,0.000068,0.000073,0.000079,0.000062,0.000083,0.000093,0.047363,0.000079,0.000062,...,0.000061,0.000048,0.000061,0.000043,0.000061,0.000047,0.415711,0.000058,0.000048,0.000700
3,0.605580,0.000099,0.000095,0.000104,0.000082,0.000120,0.134862,0.000089,0.000113,0.000095,...,0.000082,0.000066,0.000351,0.000074,0.000085,0.255952,0.001423,0.000085,0.000107,0.000244
4,0.000092,0.000173,0.000142,0.000165,0.000099,0.000175,0.145124,0.000115,0.000156,0.000227,...,0.000101,0.000080,0.001040,0.475355,0.000116,0.000205,0.000111,0.000104,0.375942,0.000106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,0.055672,0.000164,0.074794,0.000203,0.000722,0.000229,0.000121,0.000385,0.000273,0.000173,...,0.000142,0.000133,0.000222,0.633473,0.000136,0.075541,0.000220,0.156444,0.000142,0.000192
308,0.018053,0.000068,0.000168,0.000079,0.044646,0.000107,0.000096,0.215550,0.305871,0.000074,...,0.000060,0.000068,0.000059,0.000041,0.414509,0.000047,0.000074,0.000069,0.000047,0.000064
309,0.079175,0.000089,0.298401,0.000176,0.000075,0.000185,0.000096,0.000138,0.553955,0.000071,...,0.000104,0.047959,0.000076,0.002911,0.000085,0.000066,0.015942,0.000064,0.000066,0.000131


In [39]:
df_results_spotiphy

,value
metric,
mean_spot_pcc,0.057838
mean_spot_spearman,0.061815
mean_type_pcc,0.038263
mean_type_spearman,0.024552
overall_mse,0.037374
overall_rmse,0.193323
mean_jsd,0.836403
mean_ssim,0.619639
mean_moran_pred,0.014187


In [478]:
# RCTD
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/RCTD_results/RCTD_weights.csv",sep=',',header=0,index_col=0)
# 将列名中的 '/' 替换为 ' ' (空格)
gtcol =A_gt_deconv.columns.str.replace('/', '_', regex=False)
gtcol=gtcol.str.replace(' ', '_', regex=False)
pred_A=pred_A[gtcol]
pred_A=pred_A[gtcol]

pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_RCTD = evaluate_deconvolution(pred_A, A_gt_deconv.loc[pred_A.index].values,coords=spot_clean.loc[pred_A.index])

In [479]:
df_results_RCTD

,value
metric,
mean_spot_pcc,0.55592
mean_spot_spearman,0.271094
mean_type_pcc,0.498146
mean_type_spearman,0.239722
overall_mse,0.014479
overall_rmse,0.120327
mean_jsd,0.42918
mean_ssim,0.813129
mean_moran_pred,0.042886


In [480]:
#CIBERSORTx
# RCTD
pred_A=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/CIBERSORTx_Merfish.txt",sep='\t',header=0,index_col=0)
gtcol =A_gt_deconv.columns.str.replace('/', '_', regex=False)
gtcol=gtcol.str.replace(' ', '_', regex=False)
A_gt_deconv1=A_gt_deconv.copy()
A_gt_deconv1.index='X'+ A_gt_deconv1.index.astype(str)

In [481]:

pred_A=pred_A[gtcol]

pred_A = pred_A.div(pred_A.sum(axis=1), axis=0)
df_results_CIBERSORTx = evaluate_deconvolution(pred_A, A_gt_deconv1.values,coords=spot_clean)

In [482]:
# spatialDLWS
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/spatialDWLS/spatialDWLS_dec_zztry.csv',sep=',',index_col=0)
A_dev_pred=A_dev_pred[A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_')]
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_spatialDLWS = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.values,coords=spot_clean)

In [483]:
df_results_spatialDLWS

,value
metric,
mean_spot_pcc,0.30544
mean_spot_spearman,0.284624
mean_type_pcc,0.298108
mean_type_spearman,0.237899
overall_mse,0.028147
overall_rmse,0.16777
mean_jsd,0.620419
mean_ssim,0.683181
mean_moran_pred,0.02388


In [484]:
A_gt_deconv

,Astro,Endo,L2/3 IT,L4/5 IT,L5 ET,L5 IT,L5/6 NP,L6 CT,L6 IT,L6 IT Car3,...,OPC,Oligo,PVM,Peri,Pvalb,SMC,Sncg,Sst,VLMC,Vip
spot_id,,,,,,,,,,,,,,,,,,,,,
0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
1,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
2,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,1.00,0.0
4,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,0.000000,0.0,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.2,0.00,0.0
308,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.75,0.0
309,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.500000,0.00,0.5,0.0,0.0,0.0,0.0,0.0,0.00,0.0


In [485]:
#cytospace
A_dev_pred=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/cytospace_results/fractional_abundances_by_spot.csv',sep=',',header=0,index_col=0)
A_dev_pred = A_dev_pred.reindex(
    columns=A_gt_deconv.columns.str.replace('/', '_').str.replace(' ', '_'),
    fill_value=0
)
row_sums = A_dev_pred.sum(axis=1)
A_dev_pred = A_dev_pred.div(row_sums,axis=0)
A_dev_pred = A_dev_pred.fillna(0)
df_results_cytospace = evaluate_deconvolution(A_dev_pred.values, A_gt_deconv.loc[A_dev_pred.index].values,coords=spot_clean.loc[A_dev_pred.index])

In [486]:
df_results_cytospace

,value
metric,
mean_spot_pcc,0.391354
mean_spot_spearman,0.37123
mean_type_pcc,0.317016
mean_type_spearman,0.275033
overall_mse,0.024277
overall_rmse,0.15581
mean_jsd,0.554883
mean_ssim,0.694604
mean_moran_pred,0.031762


In [487]:
df_all = pd.concat(
    [df_results_ours, df_results_novosparc, df_results_tangram,
     df_results_spotiphy, df_results_RCTD, df_results_cytospace, df_results_spatialDLWS,df_results_CIBERSORTx],
    axis=1,
    keys=['scMagNet', 'novoSpaRc', 'Tangram', 'Spotiphy', 'RCTD','CytoSPACE', 'SpatialDWLS','CIBERSORTx']
)

In [488]:
df_all.T.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/dec_result.txt',sep='\t')

In [489]:
df_all

,scMagNet,novoSpaRc,Tangram,Spotiphy,RCTD,CytoSPACE,SpatialDWLS,CIBERSORTx
,value,value,value,value,value,value,value,value
metric,,,,,,,,
mean_spot_pcc,0.659304,0.416815,0.579465,0.057838,0.55592,0.391354,0.30544,0.130586
mean_spot_spearman,0.530715,0.299346,0.363064,0.061815,0.271094,0.37123,0.284624,0.078852
mean_type_pcc,0.604778,0.321944,0.524996,0.038263,0.498146,0.317016,0.298108,0.133407
mean_type_spearman,0.528534,0.246264,0.320099,0.024552,0.239722,0.275033,0.237899,0.085511
overall_mse,0.012319,0.018399,0.015127,0.037374,0.014479,0.024277,0.028147,0.028371
overall_rmse,0.110992,0.135642,0.122993,0.193323,0.120327,0.15581,0.16777,0.168438
mean_jsd,0.365386,0.57628,0.458862,0.836403,0.42918,0.554883,0.620419,0.756632
mean_ssim,0.789797,0.720816,0.75732,0.619639,0.813129,0.694604,0.683181,0.602922


## cell to spot mapping 

In [41]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_mapping(pred_probs, gt_matrix, spot_coords, k_list=[1,3,5,10]):
    """
    评估 cell-to-spot mapping 的指标。
    
    参数:
        pred_probs: (n_cells, n_spots) 预测概率矩阵
        gt_matrix: (n_cells, n_spots) ground truth 二值矩阵，每行一个1
        spot_coords: (n_spots, 2) spot 坐标
        k_list: list of int, Top-k 准确率的 k 值
    
    返回:
        metrics: dict
    """
    # 确保是 numpy 数组
    pred_probs = np.asarray(pred_probs)
    gt_matrix = np.asarray(gt_matrix)
    spot_coords = np.asarray(spot_coords)
    
    n_cells, n_spots = pred_probs.shape
    assert gt_matrix.shape == (n_cells, n_spots), "形状不匹配"
    assert spot_coords.shape[0] == n_spots, "spot 坐标数量不匹配"
    
    # 真实标签：每行的 argmax，如果全零则忽略
    true_labels = np.argmax(gt_matrix, axis=1)
    valid_mask = gt_matrix.sum(axis=1) > 0
    if not np.all(valid_mask):
        print(f"警告：{np.sum(~valid_mask)} 个细胞在 ground truth 中无标签，已忽略。")
    true_labels = true_labels[valid_mask]
    pred_probs_valid = pred_probs[valid_mask]
    
    n_valid = len(true_labels)
    if n_valid == 0:
        return {}
    
    # 预测标签
    pred_labels = np.argmax(pred_probs_valid, axis=1)
    
    # 1. Mapping accuracy
    mapping_acc = np.mean(pred_labels == true_labels)
    
    # 2. Top-k accuracy
    topk_acc = {}
    for k in k_list:
        if k > n_spots:
            k = n_spots
        # 获取每个细胞预测概率最高的 k 个 spot 索引
        topk_indices = np.argsort(-pred_probs_valid, axis=1)[:, :k]
        # 检查真实标签是否在其中
        correct = np.any(topk_indices == true_labels[:, None], axis=1)
        topk_acc[f'top{k}_acc'] = np.mean(correct)
    
    # 3. AUROC / AUPRC (宏平均)
    
    y_true_flat = gt_matrix.flatten()
    y_score_flat = pred_probs.flatten()# Filter out pairs where gt is 0 AND keep all
    macro_auroc = roc_auc_score(y_true_flat, y_score_flat)
    macro_auprc = average_precision_score(y_true_flat, y_score_flat)
    # 4. 坐标 MAE / RMSE
    # 真实坐标
    true_coords = spot_coords[true_labels]
    # 预测坐标（软分配：加权平均）
    # 归一化预测概率（确保每行和为1）
    pred_probs_norm = pred_probs_valid / (pred_probs_valid.sum(axis=1, keepdims=True) + 1e-12)
    pred_coords = pred_probs_norm @ spot_coords  # (n_valid, 2)
    # 欧氏距离
    dists = np.linalg.norm(true_coords - pred_coords, axis=1)
    coord_mae = np.mean(dists)
    coord_rmse = np.sqrt(np.mean(dists**2))
    
    # 汇总
    metrics = {
        'mapping_accuracy': mapping_acc,
        'coord_mae': coord_mae,
        'coord_rmse': coord_rmse,
        'macro_auroc': macro_auroc,
        'macro_auprc': macro_auprc,
    }
    metrics.update(topk_acc)
    return metrics
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    jaccard_score
)
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    jaccard_score
)
def evaluate_mapping_pairs(gt_matrix, pred_pairs,
                           cell_col="cellid", spot_col="spotname",
                           cell_order=None, spot_order=None):
    """
    Evaluate cell-to-spot mapping predictions against a binary ground truth matrix.

    Parameters
    ----------
    gt_matrix : np.ndarray or pd.DataFrame, shape (n_cells, n_spots)
        Binary ground-truth matrix. Each row should have exactly one 1.
    pred_pairs : pd.DataFrame
        Two columns: cell ID and spot name for predicted pairs.
    cell_col, spot_col : str
        Column names in pred_pairs indicating cell and spot IDs.
    cell_order, spot_order : list, optional
        Explicit row/column order for gt_matrix. If None, taken from gt_matrix.

    Returns
    -------
    pd.DataFrame with metrics.
    """
    # ---------- 1. Standardize gt_matrix ----------
    if isinstance(gt_matrix, pd.DataFrame):
        gt_df = gt_matrix.copy()
    else:
        gt_df = pd.DataFrame(gt_matrix)

    if cell_order is None:
        cell_order = list(gt_df.index)
    if spot_order is None:
        spot_order = list(gt_df.columns)

    gt_df = gt_df.reindex(index=cell_order, columns=spot_order).fillna(0)
    gt_arr = (gt_df.values > 0).astype(int)

    # ---------- 2. Build predicted binary matrix ----------
    pred_mat = np.zeros_like(gt_arr)

    # Map cell/spot IDs to row/col index
    cell_to_idx = {c: i for i, c in enumerate(cell_order)}
    spot_to_idx = {s: j for j, s in enumerate(spot_order)}

    # Counters for pairs outside gt space
    n_missing_cell = 0
    n_missing_spot = 0
    n_valid_pairs = 0

    for _, row in pred_pairs.iterrows():
        c = row[cell_col]
        s = row[spot_col]
        if c not in cell_to_idx:
            n_missing_cell += 1
            continue
        if s not in spot_to_idx:
            n_missing_spot += 1
            continue
        pred_mat[cell_to_idx[c], spot_to_idx[s]] = 1
        n_valid_pairs += 1

    # ---------- 3. Flatten to 1D labels ----------
    y_true = gt_arr.flatten()
    y_pred = pred_mat.flatten()

    # ---------- 4. Metrics ----------
    acc = accuracy_score(y_true, y_pred)

    # For binary one-vs-rest style metrics, use pos_label=1
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    # Jaccard on binary labels (equivalent to TP / (TP + FP + FN))
    jacc = jaccard_score(y_true, y_pred, pos_label=1, zero_division=0)

    # Additional counts for reference
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    results = {
        "metric": [
            "accuracy", "precision", "recall", "f1", "jaccard",
            "TP", "FP", "FN", "TN",
            "n_gt_positives", "n_pred_positives",
            "n_valid_pairs", "n_missing_cell", "n_missing_spot"
        ],
        "value": [
            acc, prec, rec, f1, jacc,
            tp, fp, fn, tn,
            int(y_true.sum()), int(y_pred.sum()),
            n_valid_pairs, n_missing_cell, n_missing_spot
        ]
    }
    return pd.DataFrame(results).set_index("metric")

In [42]:
import numpy as np

def get_entropy_uniformity(matrix):
    # 防止 log(0) 报错，加上极小值
    epsilon = 1e-12
    matrix = np.clip(matrix, epsilon, 1)
    
    N = matrix.shape[0] # 行数
    # 计算每列的熵
    entropy = -np.sum(matrix * np.log(matrix), axis=0)
    # 归一化 (除以最大熵 log(N))
    max_entropy = np.log(N)
    uniformity_score = entropy / max_entropy
    
    return uniformity_score

In [50]:
pred_A=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/ours/predicted_A_matrix-0425.csv',sep=',',header=0,index_col=0)

In [51]:

cell_spot_gt = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Merfish/spots_cells.csv',sep=',',header = 0,index_col=0)
adata_sc=sc.read('/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad')
matrix = pd.get_dummies(cell_spot_gt['spot_id']).astype(int)
A_gt=matrix.copy()
A_gt=A_gt[spot_clean.index]
A_gt_reindexed = A_gt.reindex(adata_sc.obs_names, fill_value=0)
A_gt=A_gt_reindexed.copy()

In [52]:
A_gt=A_gt.T

In [53]:
assigned_cells = pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Merfish/spots_cells.csv", header=0)
df_result = pd.crosstab(assigned_cells['spot_id'], assigned_cells['cell_id'])
spots_meta_df=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_Merfish/spots_info.csv", header=0)
df_result1 = pd.DataFrame(np.zeros((spots_meta_df.shape[0],A_gt.shape[1])), index=spots_meta_df['spot_id'], columns=pred_A.columns)
df_result1[df_result.columns]= df_result.values
df_result1.index=df_result.index


In [497]:
0.035202/((A_gt>0).sum().sum()/A_gt.shape[0]/A_gt.shape[1])

np.float64(46.80886776852792)

In [498]:
0.02745426/((A_gt>0).sum().sum()/A_gt.shape[0]/A_gt.shape[1])

np.float64(36.50652877742132)

In [499]:
0.006197348/((A_gt>0).sum().sum()/A_gt.shape[0]/A_gt.shape[1])

np.float64(8.240748907662946)

In [54]:
spatial_coords=spot_clean.copy()

In [56]:
spatial_coords

,center_x,center_y
0,3669.48413,-3977.299115
1,3769.48413,-3977.299115
2,3869.48413,-3977.299115
3,3769.48413,-3877.299115
4,3869.48413,-3877.299115
...,...,...
307,5369.48413,-1677.299115
308,5069.48413,-1577.299115
309,5169.48413,-1577.299115
310,5269.48413,-1577.299115


In [57]:
metrics_ours=evaluate_mapping(pred_A.T,df_result1.T,spatial_coords[['center_x','center_y']].values)

警告：3213 个细胞在 ground truth 中无标签，已忽略。


In [58]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, average_precision_score

# Flatten entire matrix
y_true_flat = df_result1.T.values.flatten()
y_score_flat = pred_A.T.values.flatten()

# Filter out pairs where gt is 0 AND keep all
micro_auroc = roc_auc_score(y_true_flat, y_score_flat)
micro_auprc = average_precision_score(y_true_flat, y_score_flat)

In [502]:
micro_auroc,micro_auprc

(0.8698190243208415, 0.03520237578726775)

In [503]:
metrics_ours

{'mapping_accuracy': np.float64(0.21421319796954313),
 'coord_mae': np.float64(610.7749913055771),
 'coord_rmse': np.float64(752.8970344902916),
 'macro_auroc': 0.8698190243208415,
 'macro_auprc': 0.03520237578726775,
 'top1_acc': np.float64(0.21421319796954313),
 'top3_acc': np.float64(0.3248730964467005),
 'top5_acc': np.float64(0.4010152284263959),
 'top10_acc': np.float64(0.516751269035533)}

In [504]:
metrics_ours

{'mapping_accuracy': np.float64(0.21421319796954313),
 'coord_mae': np.float64(610.7749913055771),
 'coord_rmse': np.float64(752.8970344902916),
 'macro_auroc': 0.8698190243208415,
 'macro_auprc': 0.03520237578726775,
 'top1_acc': np.float64(0.21421319796954313),
 'top3_acc': np.float64(0.3248730964467005),
 'top5_acc': np.float64(0.4010152284263959),
 'top10_acc': np.float64(0.516751269035533)}

In [60]:
#novosparc
novo = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/novosparc_normalization.npy') 

metrics_novo=evaluate_mapping(novo,df_result1.T,spatial_coords[['center_x','center_y']].values)

警告：3213 个细胞在 ground truth 中无标签，已忽略。


In [506]:
micro_auroc,micro_auprc

(0.8698190243208415, 0.03520237578726775)

In [507]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, average_precision_score

# Flatten entire matrix
y_true_flat = df_result1.T.values.flatten()
y_score_flat = novo.flatten()

# Filter out pairs where gt is 0 AND keep all
micro_auroc = roc_auc_score(y_true_flat, y_score_flat)
micro_auprc = average_precision_score(y_true_flat, y_score_flat)

In [508]:
metrics_novo

{'mapping_accuracy': np.float64(0.0751269035532995),
 'coord_mae': np.float64(659.8866646613224),
 'coord_rmse': np.float64(722.2329131737445),
 'macro_auroc': 0.7854128505244622,
 'macro_auprc': 0.00619734784156594,
 'top1_acc': np.float64(0.0751269035532995),
 'top3_acc': np.float64(0.14720812182741116),
 'top5_acc': np.float64(0.19695431472081218),
 'top10_acc': np.float64(0.3035532994923858)}

In [61]:
#tangram
import scanpy as sc
mapping_tangram=sc.read("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/tangram_mapping-Simulation.h5ad")
metrics_tangram=evaluate_mapping(mapping_tangram.X,df_result1.T,spatial_coords[['center_x','center_y']].values)

警告：3213 个细胞在 ground truth 中无标签，已忽略。


In [62]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, average_precision_score

# Flatten entire matrix
y_true_flat = df_result1.T.values.flatten()
y_score_flat = mapping_tangram.X.flatten()

# Filter out pairs where gt is 0 AND keep all
micro_auroc = roc_auc_score(y_true_flat, y_score_flat)
micro_auprc = average_precision_score(y_true_flat, y_score_flat)

In [511]:
micro_auroc,micro_auprc

(0.8311175846417129, 0.027454259934735083)

In [512]:
metrics_tangram

{'mapping_accuracy': np.float64(0.17563451776649747),
 'coord_mae': np.float64(643.7409676844663),
 'coord_rmse': np.float64(784.9455765291439),
 'macro_auroc': 0.8311175846417129,
 'macro_auprc': 0.027454259934735083,
 'top1_acc': np.float64(0.17563451776649747),
 'top3_acc': np.float64(0.29949238578680204),
 'top5_acc': np.float64(0.36852791878172586),
 'top10_acc': np.float64(0.4761421319796954)}

In [63]:
#SIMO
import pandas as pd
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/SMIO/Location_map1_zztry.csv', index_col=0)
metrics1_SIMO=evaluate_mapping_pairs(df_result1.T,df,spot_col='spot',cell_col='cell')

In [65]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_SIMO=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')

In [66]:
metrics1_Ours_SIMO

,value
metric,
accuracy,9.982524e-01
precision,8.205128e-02
recall,1.299492e-01
f1,1.005894e-01
jaccard,5.295821e-02
TP,1.280000e+02
FP,1.432000e+03
FN,8.570000e+02
TN,1.307359e+06


In [67]:
metrics1_SIMO

,value
metric,
accuracy,9.982417e-01
precision,7.756410e-02
recall,1.228426e-01
f1,9.508841e-02
jaccard,4.991749e-02
TP,1.210000e+02
FP,1.439000e+03
FN,8.640000e+02
TN,1.307352e+06


In [515]:
#cytospace
import pandas as pd
import numpy as np

# 读取数据
df = pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/cytospace_results/assigned_locations.csv')
metrics1_cytoscape=evaluate_mapping_pairs(df_result1.T,df,spot_col='SpotID',cell_col='OriginalCID')


In [516]:
df.shape

(1403, 6)

In [517]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_cytospace=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [518]:
# CellTrek
matrix=pd.read_csv('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CellTreck_assignment.txt',sep=' ',header=0,index_col=0)
matrix.index=[str0[1:] for str0 in matrix.index]
matrix.columns = matrix.columns.astype(int)
# Option 1: Using stack (works for both sparse and dense)
stacked = matrix.stack()
pairs = stacked[stacked >0].reset_index()
pairs.columns = ['cellid', 'spotname','value']
metrics1_CellTrek=evaluate_mapping_pairs(df_result1.T,pairs,spot_col='spotname',cell_col='cellid')


In [519]:
len(set(pairs['cellid'].values)&set(df_result1.columns))

1779

In [520]:
pairs.shape

(1779, 3)

In [521]:
import pandas as pd
import numpy as np

N = pairs.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_CellTrek=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [522]:
# CMAP
matrix=pd.read_csv("/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CMAP/cell_to_spot.txt",sep=' ')
#CMAp
# CellTrek
# Option 1: Using stack (works for both sparse and dense)

matrix.columns = matrix.columns.astype(int)
stacked = matrix.stack()
pairs = stacked[stacked >0].reset_index()
pairs.columns = ['cellid', 'spotname','value']
metrics1_CMAP=evaluate_mapping_pairs(df_result1.T,pairs,spot_col='spotname',cell_col='cellid')

In [523]:
import pandas as pd
import numpy as np

N = pairs.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_CMAP=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [524]:
# spaTrio
df = pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/spaTrio/output.csv', sep=',',index_col=0)
metrics1_SpaTrio=evaluate_mapping_pairs(df_result1.T,df,spot_col='spot',cell_col='cell')

In [525]:
df.shape

(1560, 9)

In [526]:
metrics1_SpaTrio

,value
metric,
accuracy,9.982417e-01
precision,7.756410e-02
recall,1.228426e-01
f1,9.508841e-02
jaccard,4.991749e-02
TP,1.210000e+02
FP,1.439000e+03
FN,8.640000e+02
TN,1.307352e+06


In [527]:
import pandas as pd
import numpy as np

N = df.shape[0]  # 你要选的数量

# pred_A: DataFrame, index=spot, columns=cell
stacked = pred_A.stack()                    # 变成 MultiIndex Series
top_n = stacked.nlargest(N)                 # 全局最大的 N 个
df_top = top_n.reset_index()
df_top.columns = ['spot', 'cell', 'value']
metrics1_Ours_spaTrio=evaluate_mapping_pairs(df_result1.T,df_top,spot_col='spot',cell_col='cell')


In [528]:
import pandas as pd
import numpy as np

# 1. 将所有指标的字典放入一个列表中
metrics_list = [
    metrics_ours, 
    metrics_tangram, 
    metrics_novo, 
]

# 2. 定义与列表顺序对应的标签名称
method_names = [
    "ours", 
    "Tangram", 
    "NovoSpa", 
]
# 3. 使用列表推导式，将字典批量转换为带有 index=[0] 的单行 DataFrame
# 这样每个 DataFrame 都是 1行 x N列 的结构，可以直接横向拼接
df_list = [pd.DataFrame(m, index=[0]) for m in metrics_list]

# 4. 横向合并 (axis=1)，并使用 keys 参数添加方法标签作为多级索引
metrics_all = pd.concat(df_list, axis=0, keys=method_names)

metrics_all.index=  [f"{method}" for method, metric in metrics_all.T.columns]

metrics_all.T.to_csv('cell_to_spot_result.csv')

In [529]:
metrics_all

,mapping_accuracy,coord_mae,coord_rmse,macro_auroc,macro_auprc,top1_acc,top3_acc,top5_acc,top10_acc
ours,0.214213,610.774991,752.897034,0.869819,0.035202,0.214213,0.324873,0.401015,0.516751
Tangram,0.175635,643.740968,784.945577,0.831118,0.027454,0.175635,0.299492,0.368528,0.476142
NovoSpa,0.075127,659.886665,722.232913,0.785413,0.006197,0.075127,0.147208,0.196954,0.303553


In [530]:
import pandas as pd
import numpy as np

# 1. 将所有指标的字典放入一个列表中
metrics_list = [
    metrics1_cytoscape, 
    metrics1_Ours_cytospace, 
    metrics1_CellTrek,
    metrics1_Ours_CellTrek, 
    metrics1_CMAP, 
    metrics1_Ours_CMAP, 
    metrics1_SIMO, 
    metrics1_Ours_SIMO, 
    metrics1_SpaTrio, 
    metrics1_Ours_spaTrio
]

# 2. 定义与列表顺序对应的标签名称
method_names = [
    "CytoSPACE", 
    'Ours0',
    "CellTrek", 
    'Ours1',
    "CMAP", 
    'Ours2',
    "SIMO", 
    'Ours3',
    "SpaTrio",
    'Ours4',
]
# 3. 使用列表推导式，将字典批量转换为带有 index=[0] 的单行 DataFrame
# 这样每个 DataFrame 都是 1行 x N列 的结构，可以直接横向拼接

# 4. 横向合并 (axis=1)，并使用 keys 参数添加方法标签作为多级索引
metrics_all = pd.concat(metrics_list, axis=1, keys=method_names)

# 5. (可选) 扁平化列名，方便后续导出为 Excel 或画图


In [531]:
metrics_all.to_csv('cell_asignment.txt',sep='\t')

In [532]:
metrics_all.values[1:5,1]-metrics_all.values[1:5,0]

array([0.03920171, 0.05583756, 0.04606365, 0.0249725 ])

In [533]:
metrics_all.values[1:5,3]-metrics_all.values[1:5,2]

array([0.05733558, 0.1035533 , 0.07380608, 0.03923754])

In [534]:
metrics_all.values[1:5,5]-metrics_all.values[1:5,4]

array([0.05853358, 0.0964467 , 0.07285276, 0.03885886])

In [535]:
metrics_all.values[1:5,7]-metrics_all.values[1:5,6]

array([0.00448718, 0.0071066 , 0.00550098, 0.00304072])

In [536]:
metrics_all

,CytoSPACE,Ours0,CellTrek,Ours1,CMAP,Ours2,SIMO,Ours3,SpaTrio,Ours4
,value,value,value,value,value,value,value,value,value,value
metric,,,,,,,,,,
accuracy,9.982791e-01,9.983631e-01,9.979386e-01,9.980943e-01,9.980623e-01,9.982073e-01,9.982417e-01,9.982524e-01,9.982417e-01,9.982524e-01
precision,4.775481e-02,8.695652e-02,1.798763e-02,7.532322e-02,2.156500e-02,8.009858e-02,7.756410e-02,8.205128e-02,7.756410e-02,8.205128e-02
recall,6.802030e-02,1.238579e-01,3.248731e-02,1.360406e-01,3.553299e-02,1.319797e-01,1.228426e-01,1.299492e-01,1.228426e-01,1.299492e-01
f1,5.611390e-02,1.021776e-01,2.315485e-02,9.696093e-02,2.684049e-02,9.969325e-02,9.508841e-02,1.005894e-01,9.508841e-02,1.005894e-01
jaccard,2.886687e-02,5.383936e-02,1.171303e-02,5.095057e-02,1.360280e-02,5.246166e-02,4.991749e-02,5.295821e-02,4.991749e-02,5.295821e-02
TP,6.700000e+01,1.220000e+02,3.200000e+01,1.340000e+02,3.500000e+01,1.300000e+02,1.210000e+02,1.280000e+02,1.210000e+02,1.280000e+02
FP,1.336000e+03,1.281000e+03,1.747000e+03,1.645000e+03,1.588000e+03,1.493000e+03,1.439000e+03,1.432000e+03,1.439000e+03,1.432000e+03
FN,9.180000e+02,8.630000e+02,9.530000e+02,8.510000e+02,9.500000e+02,8.550000e+02,8.640000e+02,8.570000e+02,8.640000e+02,8.570000e+02


In [537]:
data0=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/sc_expression.tsv',sep='\t',header=0,index_col=0)
label=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/cell_type_labels.tsv',sep='\t',header=0,index_col=0)

In [538]:
(label.index==data0.columns).sum()

np.int64(4198)

In [539]:
data0.columns=label['CellType'].values
data0.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/sc_expression_0.tsv',sep='\t')
data1=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/st_expression.tsv',sep='\t',header=0,index_col=0)
data1.columns='X'+data1.columns
data1.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/Cytospace/st_expression_0.tsv',sep='\t')

In [540]:
data0=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/sc_expression.tsv',sep='\t',header=0,index_col=0)
label=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/cell_type_labels.tsv',sep='\t',header=0,index_col=0)

In [541]:
(label.index==data0.columns).sum()

np.int64(2489)

In [542]:

import numpy as np

labels = label['CellType'].values
labels_clean = np.char.replace(labels.astype(str), ' ', '_')
print(labels_clean[:5])

['Visceral_muscle' 'Trachea' 'Epidermis' 'Foregut'
 'Stomatogastric_nervous_system']


In [543]:
data0.columns=labels_clean
#data0.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/sc_expression_stereo.tsv',sep='\t')
# Round all numeric columns and convert to int
data0_int = data0.copy()

data0_int = data0_int.astype(int)

data0_int.to_csv(
    '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/sc_expression_stereo.tsv',
    sep='\t'
)


In [544]:
data1=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/st_expression.tsv',sep='\t',header=0,index_col=0)


In [545]:
data1.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/decon/st_expression_stereo.tsv',sep='\t')

In [549]:
data0=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/sc_expression.tsv',sep='\t',header=0,index_col=0)
label=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/cell_type_labels.tsv',sep='\t',header=0,index_col=0)

In [550]:
data0.columns = label['CellType'].values

In [551]:
data0_sample = data0.sample(n=5000, axis=1, random_state=123)

In [553]:
data0_sample.to_csv(
    '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/Xenium_Sim/sc_expression_Xen_5000.tsv',
    sep='\t'
)

In [2]:
data0= pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/sc_expression.tsv',sep='\t',header=0,index_col=0)
label=pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/cell_type_labels.tsv',sep='\t',header=0,index_col=0)

In [3]:
(label.index==data0.columns).sum()

np.int64(19979)

In [4]:
markers = np.load('/home/qyyuan/project/ST_CP/JE/Tangram/20k_markers.npy')
common_genes = markers.tolist()

In [5]:
data0=data0.loc[common_genes,:]

In [6]:
data0.columns = label['cell_type'].values

In [565]:
data0.to_csv(
    '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/sc_expression_MB.tsv',
    sep='\t'
)

In [7]:
# 使用 pandas 的 groupby，并一次性采样所有需要的内容
col_series = pd.Series(data0.columns)
groups = col_series.groupby(col_series).indices

# 对每个组进行采样
np.random.seed(123)

# 第一轮：每个类别至少 10 个
first_round = []
for idx_list in groups.values():
    n = min(10, len(idx_list))
    first_round.extend(np.random.choice(idx_list, n, replace=False))

# 第二轮：从剩余列中填充
remaining = 2000 - len(first_round)
if remaining > 0:
    all_idx = np.arange(len(data0.columns))
    mask = np.ones(len(all_idx), dtype=bool)
    mask[first_round] = False
    pool = all_idx[mask]
    extra = np.random.choice(pool, min(remaining, len(pool)), replace=False)
    final = np.concatenate([first_round, extra])
else:
    final = np.array(first_round)

data0_sample = data0.iloc[:, final]

In [9]:
data0_sample.to_csv(
    '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/sc_expression_MB_2000.tsv',
    sep='\t'
)

In [10]:
data1= pd.read_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/st_expression.tsv',sep='\t',header=0,index_col=0)


In [12]:
data1=data1.loc[list(set(data1.index)&set(common_genes)),:]

In [13]:
data1.to_csv(
    '/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/st_expression_MB.tsv',
    sep='\t'
)

In [14]:
sc_data_path = "/home/qyyuan/project/ST_CP/JE/method/data/scRNA_subsampled_20k.h5ad"
st_data_path = "/home/qyyuan/project/ST_CP/JE/compare/groundtruth/NM_yangjy_X/data/Visium_FAD.h5ad"
import scanpy as sc
adatasc = sc.read(sc_data_path)

/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [20]:
sc.pp.filter_genes(adatasc, min_cells=3)

# 过滤基因数过少的细胞（可选，视数据而定）
sc.pp.filter_cells(adatasc, min_genes=200)


In [22]:
sc.pp.highly_variable_genes(
    adatasc,
    n_top_genes=10000,
    flavor='seurat_v3',   # 直接在原始 counts 上运行
    subset=True
)

In [25]:
df_hvg = pd.DataFrame(
    adatasc.X.toarray() if hasattr(adatasc.X, "toarray") else adatasc.X,
    index=adatasc.obs_names,
    columns=adatasc.var_names
)

In [33]:
df_hvg=df_hvg.T

In [44]:
df_hvg.columns=label['cell_type'].values

In [45]:
# 使用 pandas 的 groupby，并一次性采样所有需要的内容
col_series = pd.Series(df_hvg.columns)
groups = col_series.groupby(col_series).indices

# 对每个组进行采样
np.random.seed(123)

# 第一轮：每个类别至少 10 个
first_round = []
for idx_list in groups.values():
    n = min(10, len(idx_list))
    first_round.extend(np.random.choice(idx_list, n, replace=False))

# 第二轮：从剩余列中填充
remaining = 2000 - len(first_round)
if remaining > 0:
    all_idx = np.arange(len(df_hvg.columns))
    mask = np.ones(len(all_idx), dtype=bool)
    mask[first_round] = False
    pool = all_idx[mask]
    extra = np.random.choice(pool, min(remaining, len(pool)), replace=False)
    final = np.concatenate([first_round, extra])
else:
    final = np.array(first_round)

data0_sample = df_hvg.iloc[:, final]

In [47]:
data0_sample.to_csv('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/jiyuanYang/deconvolution/Cytospace/sc_expression_MB_2000.tsv',sep='\t')

In [67]:
## stalocator
import numpy as np
cell_loc=np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/STALocator/map_A_sc.npy')
spot_loc = np.load('/data/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/STALocator/map_B_ST.npy')

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
import pandas as pd

# 假设 spot_loc 和 cell_loc 是 numpy array，形状分别为 (477, 2) 和 (16068, 2)
spot_loc = np.asarray(spot_loc)
cell_loc = np.asarray(cell_loc)

n_cells = cell_loc.shape[0]
n_spots = spot_loc.shape[0]
k = 10  # 每个 cell 找最近的 10 个 spot
# 1. 构建 KDTree
tree = cKDTree(spot_loc)
# 2. 查询每个 cell 最近的 k 个 spot
# distances: (n_cells, k), indices: (n_cells, k)
distances, indices = tree.query(cell_loc, k=k)
# 3. 权重：top1=10, top2=9, ..., top10=1
weights = np.arange(k, 0, -1)  # [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]

# 4. 构建稀疏矩阵 (n_cells × n_spots)
rows = np.repeat(np.arange(n_cells), k)
cols = indices.ravel()
vals = np.tile(weights, n_cells)

M = csr_matrix((vals, (rows, cols)), shape=(n_cells, n_spots))

# 如果需要稠密 DataFrame：
# df = pd.DataFrame(M.toarray())

print(M.shape)   # (16068, 477)
print(M.nnz)     # 16068 * 10 = 160680 个非零值

(4198, 312)
41980


In [69]:
metrics_STALocator=evaluate_mapping(M.toarray(),df_result1.T,spatial_coords[['row','col']].values)

警告：3213 个细胞在 ground truth 中无标签，已忽略。


In [70]:
metrics_STALocator

{'mapping_accuracy': np.float64(0.003045685279187817),
 'coord_mae': np.float64(710.6121378070613),
 'coord_rmse': np.float64(778.209645897607),
 'macro_auroc': 0.5038018308732692,
 'macro_auprc': 0.0007644082554119482,
 'top1_acc': np.float64(0.003045685279187817),
 'top3_acc': np.float64(0.016243654822335026),
 'top5_acc': np.float64(0.023350253807106598),
 'top10_acc': np.float64(0.03959390862944162)}